In [112]:
import pandas as pd
import numpy as np

In [121]:
Messy_data=pd.read_csv('messy_data.csv')
Messy_data.head(2)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,ASTHMA,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com


#**This data has following issues --- Manual checking

    #**1--Inconsistent categorical casing/spacing — gender has Male
    #MALE, male, M, " Male", Female, FEMALE, female, F, Other, O
    
    #**2--Typos & inconsistent spelling, condition has Diabetes, 
    #diabetees, DIABETES, Diabetis, Hypertension, Hypertention, 
    #Arthritis, Arthritus, Migraine, Migrain, Asma

    #**3--Inconsistent date formats — admission_date mixes 2024-01-05, 
    #01/05/2024, Jan 05 2024, 05-Jan-24, 01-05-2024

    #**4--Inconsistent phone formats — (555) 123-4567, 555-123-4567, 
    #5551234567, 555.123.4567

    #**5--Inconsistent boolean representations — is_smoker has Yes/No
    #/Y/N/1/0/True/False in various cases

    #**6--Inconsistent name casing — some last_name values are ALL CAPS, 
    #some emails are uppercase

    #**7--Multiple missing-value representations — NaN, '', 'N/A', 'n/a',        'Unknown', 'unknown', 'null', 'NULL', '--', '?' all mean "missing" but        pandas won't recognize most of them automatically

    #**8--Mixed-type numeric column — age is stored as text: "45", "45 yrs",     "45.0", "45 years old"

    #**9--Impossible/outlier values — negative ages, age of 300, negative or     zero weight, weight of 500

    #**10--Currency stored as text — treatment_cost mixes "$1,200.50",           "1200.50", and plain numbers — can't do math on it as-is

    #**11--Extra whitespace padding — leading/trailing spaces in first_name,     patient_id, condition

    #**12--Duplicate rows — exact duplicates were inserted Near-duplicate        rows — same person, but with different casing/spacing making them look       different (a classic dedup challenge)



#*Programmatic cleaning

#**1--Inconsistent categorical casing/spacing — gender has Male, MALE, male, M, " Male", Female, FEMALE, female, F, Other, O

In [46]:
gender_map={'M': 'Male', ' Male': 'Male', 'MALE':'Male', 'male':'Male', 'FEMALE':'Female', 
            ' Female':'Female', 'female':'Female', 'F':'Female', 'O':'Other', 'Other':'Other'}

# Normalizing the dict keys the same way you normalize the data
gender_map_clean = {k.strip().lower(): v for k, v in gender_map.items()}

Messy_data['gender']=Messy_data['gender'].astype(str).str.strip().str.lower().map(gender_map_clean)
Messy_data.head(20)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,migraine,04-28-2023,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,ASTHMA,24-May-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,DIABETES,Jul 13 2023,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,56 yrs,Hypertension,2024-08-05,8964449377,68.7 kg,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3,Diabetis,2023-09-10,2872479217,80.0,N,"$1,880.02",wei.jones272@example.com
6,PT900155,Elizabeth,Brown,Female,41.0,HYPERTENSION,Sep 13 2024,(386) 585-4192,101.4 kg,False,1290.19,elizabeth.brown155@example.com
7,PT900152,Amara,MILLER,Female,41.0,ARTHRITIS,Aug 08 2023,3835219110,108.8,TRUE,"$1,398.36",amara.miller152@example.com
8,PT900165,David,Smith,Female,43 years old,MIGRAINE,07-19-2023,8732839356,0,N,"1,538.11",david.smith165@example.com
9,PT900175,Wei,Patel,Male,38,asthma,05-20-2023,591.905.8256,107.5,FALSE,"4,944.31",wei.patel175@example.com


#**2--Typos & inconsistent spelling, condition has Diabetes, diabetees,            DIABETES, Diabetis, Hypertension, Hypertention, Arthritis, Arthritus,        Migraine, Migrain, Asma

In [47]:
Messy_data['condition'].unique()

<StringArray>
[     'migraine',        'ASTHMA',      'DIABETES',        'asthma',
 ' Hypertension',      'Diabetis',  'HYPERTENSION',     'ARTHRITIS',
      'MIGRAINE',     'Arthritis',  'hypertension',      'Migraine',
   'Arthritis  ',       'Asthma ',       'Migrain',  'Hypertension',
        'Asthma',      'diabetes',     'Arthritus',             nan,
     'diabetees',          'Asma',   '  Diabetes ',  'Hypertention',
     'arthritis',      'Diabetes',       'Unknown',    ' Migraine ',
            '--',       'unknown',             '?']
Length: 31, dtype: str

In [74]:
condition_map={'migraine':'Migraine', 'ASTHMA':'Asthma', 'DIABETES':'Diabetes', 'asthma':'Asthma',
 ' Hypertension':'Hypertension', 'Diabetis':'Diabetes', 'HYPERTENSION':'Hypertension', 'ARTHRITIS':'Arthritis',
      'MIGRAINE':'Migraine', 'hypertension':'Hypertension', 'Arthritis  ':'Arthritis', 'Asthma ':'Asthma', 
        'Migrain':'Migraine', 'diabetes':'Diabetes', 'Arthritus':'Arthritis', 'nan':'NaN',
     'diabetees':'Diabetes', 'Asma':'Asthma', '  Diabetes ':'Diabetes', 'Hypertention':'Hypertension',
     'arthritis':'Arthritis', 'Migraine ':'Migraine'}

Messy_data['condition']=Messy_data['condition'].astype(str).str.strip().str.lower().map(condition_map)
Messy_data.head(20)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,04-28-2023,7257273491,105.9,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83.0,Asthma,24-May-24,531.879.2068,105.3,N,510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16.0,Diabetes,Jul 13 2023,438-180-8444,65.4,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,530-442-7454,60.5,1,3317.58,elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,unknown,Hypertension,2024-08-05,8964449377,68.7,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3.0,NaN,2023-09-10,2872479217,80.0,N,1880.02,wei.jones272@example.com
6,PT900155,Elizabeth,Brown,Female,41.0,Hypertension,Sep 13 2024,(386) 585-4192,101.4,False,1290.19,elizabeth.brown155@example.com
7,PT900152,Amara,MILLER,Female,41.0,Arthritis,Aug 08 2023,3835219110,108.8,TRUE,1398.36,amara.miller152@example.com
8,PT900165,David,Smith,Female,unknown,Migraine,07-19-2023,8732839356,0.0,N,1538.11,david.smith165@example.com
9,PT900175,Wei,Patel,Male,38.0,Asthma,05-20-2023,591.905.8256,107.5,FALSE,4944.31,wei.patel175@example.com


In [75]:
Messy_data['condition'].unique()

<StringArray>
['Migraine', 'Asthma', 'Diabetes', 'Hypertension', nan, 'Arthritis']
Length: 6, dtype: str

#**3--Inconsistent date formats — admission_date mixes 2024-01-05, 
#01/05/2024, Jan 05 2024, 05-Jan-24, 01-05-2024

In [76]:
Messy_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   patient_id      429 non-null    str   
 1   first_name      429 non-null    str   
 2   last_name       429 non-null    str   
 3   gender          429 non-null    str   
 4   age             429 non-null    object
 5   condition       328 non-null    str   
 6   admission_date  429 non-null    str   
 7   phone           429 non-null    str   
 8   weight          429 non-null    object
 9   is_smoker       429 non-null    str   
 10  treatment_cost  429 non-null    object
 11  email           429 non-null    str   
dtypes: object(3), str(9)
memory usage: 40.3+ KB


In [77]:
Messy_data['admission_date']=pd.to_datetime(Messy_data['admission_date'], format='mixed', errors='coerce')

In [11]:
Messy_data.head(4)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,2023-04-28,7257273491,105.9 kg,N,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83,Asthma,2024-05-24,531.879.2068,105.3,N,$510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,WILLIAMS,Male,16,Diabetes,2023-07-13,438-180-8444,65.4 kg,1,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,530-442-7454,60.5,1,"3,317.58",elizabeth.davis30@example.com


In [78]:
Messy_data['admission_date'].unique()

<DatetimeArray>
['2023-04-28 00:00:00', '2024-05-24 00:00:00', '2023-07-13 00:00:00',
 '2024-04-11 00:00:00', '2024-08-05 00:00:00', '2023-09-10 00:00:00',
 '2024-09-13 00:00:00', '2023-08-08 00:00:00', '2023-07-19 00:00:00',
 '2023-05-20 00:00:00',
 ...
 '2024-06-30 00:00:00', '2025-04-07 00:00:00', '2024-09-24 00:00:00',
 '2025-04-26 00:00:00', '2025-01-08 00:00:00', '2023-01-16 00:00:00',
 '2024-06-03 00:00:00', '2023-08-02 00:00:00', '2024-07-20 00:00:00',
 '2023-11-05 00:00:00']
Length: 322, dtype: datetime64[us]

In [79]:
print(Messy_data['admission_date'].isna().sum())

4


#**4--Inconsistent phone formats — (555) 123-4567, 555-123-4567, 
#5551234567, 555.123.4567


In [102]:
Messy_data['phone'].unique()

<StringArray>
[    '7257273491',   '531.879.2068',   '438-180-8444',   '530-442-7454',
     '8964449377',     '2872479217', '(386) 585-4192',     '3835219110',
     '8732839356',   '591.905.8256',
 ...
   '870.931.4605', '(898) 640-2091',     '7327126201',     '5597898955',
   '682.811.8819',     '8648157761',   '292-836-9378',     '4685195066',
   '666.506.1397',   '867-818-4145']
Length: 384, dtype: str

In [104]:
Messy_data['phone']=Messy_data['phone'].astype(str).str.replace(r'\D', '', regex=True)
Messy_data['phone']

0      7257273491
1      5318792068
2      4381808444
3      5304427454
4      8964449377
          ...    
424    2928369378
425    4685195066
426    6665061397
427    8678184145
428    9486643791
Name: phone, Length: 429, dtype: str

In [107]:
Messy_data['phone'].unique()

<StringArray>
['7257273491', '5318792068', '4381808444', '5304427454', '8964449377',
 '2872479217', '3865854192', '3835219110', '8732839356', '5919058256',
 ...
 '8709314605', '8986402091', '7327126201', '5597898955', '6828118819',
 '8648157761', '2928369378', '4685195066', '6665061397', '8678184145']
Length: 382, dtype: str

#**5--Inconsistent boolean representations — is_smoker has Yes/No
#/Y/N/1/0/True/False in various cases

In [80]:
Messy_data['is_smoker'].unique()

<StringArray>
[      'N',       '1',      'No',   'False',    'TRUE',   'FALSE',      'no',
     'YES',     'Yes',    'True',    'true',      'NO',       'Y',      '--',
 'unknown',   'false',     'yes',       '0',       '?', 'Unknown']
Length: 20, dtype: str

In [108]:
smoking_map={'N':'No', '1':'Yes', 'No':'No', 'False':'No', 'TRUE':'Yes', 'FALSE':'No', 'no':'No',
     'YES':'Yes', 'nan':'NaN', 'Yes':'Yes', 'True':'Yes', 'true':'Yes', 'NO':'No', 'Y':'Yes', 'false':'No', 'yes':'Yes', '0':'No'}

smoking_clean={k.strip().lower():v for k, v in smoking_map.items()}

Messy_data['is_smoker']=Messy_data['is_smoker'].astype(str).str.strip().str.lower().map(smoking_clean)
Messy_data.head(6)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,2023-04-28,7257273491,105.9,No,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83.0,Asthma,2024-05-24,5318792068,105.3,No,510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,Williams,Male,16.0,Diabetes,2023-07-13,4381808444,65.4,Yes,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,5304427454,60.5,Yes,3317.58,elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,unknown,Hypertension,2024-08-05,8964449377,68.7,No,1397.13,sofia.brown360@example.com
5,PT900272,Wei,Jones,Male,3.0,unknown,2023-09-10,2872479217,80.0,No,1880.02,wei.jones272@example.com


#**6--Inconsistent name casing — some last_name values are ALL CAPS, 


In [82]:
Messy_data['last_name'].unique()

<StringArray>
[ 'Williams',    'Nguyen',  'WILLIAMS',     'Davis',     'Brown',     'Jones',
    'MILLER',     'Smith',     'Patel',  'Martinez',       'Kim',      'Chen',
    'Garcia',    'Miller',       'Lee', 'Rodriguez',      'Khan',   'JOHNSON',
    'GARCIA',   'Johnson',   'unknown',    'NGUYEN',     'BROWN',      'CHEN',
     'JONES',       'LEE',  'MARTINEZ', 'RODRIGUEZ',     'PATEL',     'SMITH',
     'DAVIS']
Length: 31, dtype: str

In [83]:
Messy_data['last_name']=Messy_data['last_name'].astype(str).str.strip().str.title()
Messy_data['last_name']

0      Williams
1        Nguyen
2      Williams
3         Davis
4         Brown
         ...   
424        Chen
425    Williams
426       Jones
427        Khan
428       Jones
Name: last_name, Length: 429, dtype: str

In [85]:
Messy_data['last_name'].unique()

<StringArray>
[ 'Williams',    'Nguyen',     'Davis',     'Brown',     'Jones',    'Miller',
     'Smith',     'Patel',  'Martinez',       'Kim',      'Chen',    'Garcia',
       'Lee', 'Rodriguez',      'Khan',   'Johnson',   'Unknown']
Length: 17, dtype: str

###some emails are uppercase

In [86]:
Messy_data['email'].unique()

<StringArray>
['hiroshi.williams295@example.com',      'sofia.nguyen75@example.com',
  'carlos.williams177@example.com',   'elizabeth.davis30@example.com',
      'sofia.brown360@example.com',        'wei.jones272@example.com',
  'elizabeth.brown155@example.com',     'amara.miller152@example.com',
      'david.smith165@example.com',        'wei.patel175@example.com',
 ...
        'elena.khan87@example.com',   'ELENA.WILLIAMS330@EXAMPLE.COM',
       'priya.khan214@example.com',      'ahmed.patel121@example.com',
       'michael.lee20@example.com',    'elizabeth.kim188@example.com',
        'priya.chen71@example.com',     'wei.williams106@example.com',
      'david.jones270@example.com',       'DAVID.KHAN348@EXAMPLE.COM']
Length: 393, dtype: str

In [87]:
Messy_data['email']=Messy_data['email'].astype(str).str.strip().str.lower()

In [88]:
Messy_data['email'].unique()

<StringArray>
['hiroshi.williams295@example.com',      'sofia.nguyen75@example.com',
  'carlos.williams177@example.com',   'elizabeth.davis30@example.com',
      'sofia.brown360@example.com',        'wei.jones272@example.com',
  'elizabeth.brown155@example.com',     'amara.miller152@example.com',
      'david.smith165@example.com',        'wei.patel175@example.com',
 ...
        'elena.khan87@example.com',   'elena.williams330@example.com',
       'priya.khan214@example.com',      'ahmed.patel121@example.com',
       'michael.lee20@example.com',    'elizabeth.kim188@example.com',
        'priya.chen71@example.com',     'wei.williams106@example.com',
      'david.jones270@example.com',       'david.khan348@example.com']
Length: 392, dtype: str

In [109]:
Messy_data.head(4)

,patient_id,first_name,last_name,gender,age,condition,admission_date,phone,weight,is_smoker,treatment_cost,email
0,PT900295,HIROSHI,Williams,Female,51.0,Migraine,2023-04-28,7257273491,105.9,No,3960.83,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,83.0,Asthma,2024-05-24,5318792068,105.3,No,510.81,sofia.nguyen75@example.com
2,PT900177,Carlos,Williams,Male,16.0,Diabetes,2023-07-13,4381808444,65.4,Yes,875.51,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,92.0,Asthma,2024-04-11,5304427454,60.5,Yes,3317.58,elizabeth.davis30@example.com


#**7--Multiple missing-value representations — NaN, '', 'N/A', 'n/a',        
#'Unknown', 'unknown', 'null', 'NULL', '--', '?' all mean "missing" but        
#pandas won't recognize most of them automatically

In [156]:
##converting treatment_cost to float

Messy_data['treatment_cost']=Messy_data['treatment_cost'].astype(str).str.replace(r'[$,]', '', regex=True)
Messy_data['treatment_cost']=pd.to_numeric(Messy_data['treatment_cost'], errors='coerce')  # text to numeric invalid --Nan
Messy_data['treatment_cost']

0      3960.83
1       510.81
2       875.51
3      3317.58
4      1397.13
        ...   
424    4892.20
425    1228.27
426    2089.67
427     862.07
428     835.05
Name: treatment_cost, Length: 429, dtype: float64

In [157]:
Messy_data.isna().sum()

patient_id         4
first_name         4
last_name          4
gender             4
age                0
condition         15
admission_date     4
phone             15
weight             0
is_smoker         15
treatment_cost    32
email              9
dtype: int64

#**8--Mixed-type numeric column — age is stored as text: "45", "45 yrs",     "45.0", "45 years old"

In [158]:
Messy_data['age'].unique()

array([51.0, 83.0, 16.0, 92.0, 'unknown', 3.0, 41.0, 38.0, 71.0, 94.0,
       87.0, 70.0, 88.0, 45.0, 14.0, 93.0, 84.0, 36.0, 5.0, 10.0, 72.0,
       49.0, 66.0, 44.0, 65.0, 37.0, 74.0, 42.0, 33.0, 35.0, 300.0, 0.0,
       18.0, 62.0, 91.0, 46.0, 21.0, 90.0, 61.0, 29.0, 82.0, 80.0, 79.0,
       53.0, 73.0, 13.0, 57.0, 60.0, 56.0, 30.0, 78.0, 50.0, 69.0, 77.0,
       34.0, 150.0, 52.0, 59.0, 48.0, 39.0, 11.0, 28.0, 55.0, 19.0, 58.0,
       15.0, 20.0, 63.0, 67.0, 47.0, 68.0, 86.0, 26.0, 17.0, 85.0, 23.0,
       1.0, 54.0, 7.0, 64.0, 40.0, 22.0, 89.0, 12.0, 75.0, 2.0, 4.0, 6.0],
      dtype=object)

In [159]:
Messy_data['age']=Messy_data['age'].astype(str)

In [160]:
Messy_data['age']=Messy_data['age'].str.strip(r'(\d+\.?\d*)')

In [161]:
Messy_data['age'] = pd.to_numeric(Messy_data['age'], errors='coerce')

In [139]:
Messy_data['age'].dtype

dtype('float64')

In [163]:
Messy_data['weight']=Messy_data['weight'].astype(str).str.extract(r'(\d+\.?\d*)')
Messy_data['weight']=pd.to_numeric(Messy_data['weight'], errors='coerce')  # text to numeric invalid --Nan
Messy_data['weight']

0      105.9
1      105.3
2       65.4
3       60.5
4       68.7
       ...  
424     88.1
425     74.2
426     56.3
427     55.9
428     66.8
Name: weight, Length: 429, dtype: float64

In [164]:
Messy_data['weight'].unique()

array([105.9, 105.3,  65.4,  60.5,  68.7,  80. , 101.4, 108.8,   0. ,
       107.5,  81. ,  83.9,  66.6,  55.7,  69.8,  89. ,  53.7,  48.6,
        69.3,  86.3,  84.1, 500. ,  78.8,  75.2,  88.3,  67.2,  71.9,
        93.3,  63.5,  80.3,  91.1,  67.7,  87.8,  79. ,  64.2,  62.3,
        68.2,  55.1,   nan,  60.4,  75.9,  68.8,  70.2,  67. ,  68.3,
        59.3,  39.6,  74.2,  55.4,  82.3,  88.4,  79.1, 108.7,  87.9,
        84.3,  83.3,  96.5,  66.7,  73.9,  92.1,  58. ,  68.4,  58.2,
       115.5,  80.5,  67.9,  79.7,  86.8,  82.6,  90.1,  69.5,  71.3,
        62.1, 102. ,  72.1,  59.1,  58.5,  66.1,  61.7,  78.7,  71.5,
        94.6,  56.4,  94.5,  63.2,  73.5,  93.2,  79.9,  74.6,  67.8,
        81.3,  83.1,  59. ,  73.2, 100. , 112.9,  84.5,  76. ,  86.2,
        76.6,  76.2,  67.4,  46.1, 112.7,  80.6,  67.5,  62.2,  61.2,
        90.9,  72. ,  68.9,  98.3,  71.2, 100.5,  85.6, 104.3,  71.4,
        54.9,  86. ,  99.6,  69.2,  63.1,  64.8,  65.1,  77.6,  86.4,
        77.9,  97.2,

In [178]:
Messy_data_num_cols=Messy_data.select_dtypes(include='number').columns
Messy_data[Messy_data_num_cols]=Messy_data[Messy_data_num_cols].fillna('unknown')
Messy_data[Messy_data_num_cols]

,age,weight,treatment_cost
0,51.0,105.9,3960.83
1,83.0,105.3,510.81
2,16.0,65.4,875.51
3,92.0,60.5,3317.58
4,50.0,68.7,1397.13
...,...,...,...
424,30.0,88.1,4892.2
425,50.0,74.2,1228.27
426,50.0,56.3,2089.67
427,50.0,55.9,862.07


In [110]:
Messy_data_cat_cols=Messy_data.select_dtypes(include='str').columns
Messy_data[Messy_data_cat_cols]=Messy_data[Messy_data_cat_cols].fillna('unknown')
Messy_data[Messy_data_cat_cols]

,patient_id,first_name,last_name,gender,condition,phone,is_smoker,email
0,PT900295,HIROSHI,Williams,Female,Migraine,7257273491,No,hiroshi.williams295@example.com
1,PT900075,Sofia,Nguyen,Male,Asthma,5318792068,No,sofia.nguyen75@example.com
2,PT900177,Carlos,Williams,Male,Diabetes,4381808444,Yes,carlos.williams177@example.com
3,PT900030,Elizabeth,Davis,Female,Asthma,5304427454,Yes,elizabeth.davis30@example.com
4,PT900360,Sofia,Brown,Female,Hypertension,8964449377,No,sofia.brown360@example.com
...,...,...,...,...,...,...,...,...
424,PT900071,Priya,Chen,Male,Asthma,2928369378,No,priya.chen71@example.com
425,PT900106,Wei,Williams,Female,Hypertension,4685195066,No,wei.williams106@example.com
426,PT900270,David,Jones,Female,Diabetes,6665061397,Yes,david.jones270@example.com
427,PT900348,David,Khan,Male,unknown,8678184145,No,david.khan348@example.com


In [144]:
Messy_data[Messy_data_cat_cols].tail(10)

,patient_id,first_name,last_name,gender,condition,phone,is_smoker,email
419,PT900214,Priya,Khan,Male,Unknown,7327126201,0,priya.khan214@example.com
420,PT900121,Ahmed,Patel,O,migraine,5597898955,no,ahmed.patel121@example.com
421,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
422,PT900020,Michael,Lee,Male,migraine,682.811.8819,yes,michael.lee20@example.com
423,PT900188,Elizabeth,Kim,Female,ASTHMA,8648157761,false,elizabeth.kim188@example.com
424,PT900071,Priya,Chen,Male,ASTHMA,292-836-9378,False,priya.chen71@example.com
425,PT900106,Wei,Williams,female,Hypertension,4685195066,N,wei.williams106@example.com
426,PT900270,David,Jones,Female,diabetees,666.506.1397,true,david.jones270@example.com
427,PT900348,David,Khan,male,Hypertention,867-818-4145,NO,DAVID.KHAN348@EXAMPLE.COM
428,PT900102,Elena,Jones,Male,Arthritis,(948) 664-3791,Y,elena.jones102@example.com


#**9--Impossible/outlier values — negative ages, age of 300, negative 
#orzero weight, weight of 500

In [179]:

Messy_data.loc[Messy_data['age']<0, 'age']=np.nan
Messy_data.loc[(Messy_data['weight']<= 0) |(Messy_data['weight']>250), 'weight']=np.nan

In [180]:

Messy_data['age'] = Messy_data['age'].fillna(Messy_data['age'].median())
Messy_data['weight'] = Messy_data['weight'].fillna(Messy_data['weight'].median())